# USA Economy Analysis

## 03-feature-engineering: What new variables can we create?

## imports

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

# USA Economy Analysis

## 03-feature-engineering: What new variables can we create?

## imports

In [2]:
import pandas as pd
from pathlib import Path
import numpy as np

## Load df_economy

In [3]:
PROCESSED_PATH = Path("../data/processed")

df_economy= pd.read_feather(
    PROCESSED_PATH / "economy_quarterly.feather"
)

## Data observation

In [4]:
df_economy.head()

,year,quarter,year-quarter,gdp,cpi,unemployment_rate
0,1974,1,1974Q1,1491.209,47.300000,5.133333
1,1974,2,1974Q2,1530.056,48.566667,5.200000
2,1974,3,1974Q3,1560.026,49.933333,5.633333
3,1974,4,1974Q4,1599.679,51.466667,6.600000
4,1975,1,1975Q1,1616.116,52.566667,8.266667


In [5]:
df_economy.tail()

,year,quarter,year-quarter,gdp,cpi,unemployment_rate
196,2023,1,2023Q1,26813.601,301.203000,3.500000
197,2023,2,2023Q2,27063.012,303.466667,3.566667
198,2023,3,2023Q3,27610.128,306.034333,3.700000
199,2023,4,2023Q4,27956.998,308.099000,3.733333
200,2024,1,2024Q1,28269.174,309.685000,3.700000


In [6]:
df_economy.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               201 non-null    int32  
 1   quarter            201 non-null    int32  
 2   year-quarter       201 non-null    str    
 3   gdp                201 non-null    float64
 4   cpi                201 non-null    float64
 5   unemployment_rate  201 non-null    float64
dtypes: float64(3), int32(2), str(1)
memory usage: 9.2 KB


## Create growth features

Growth features measure quarter-to-quarter changes in the main economic variables.

- gdp_growth: Quarterly GDP growth rate
- cpi_growth: Quarterly CPI growth rate
- unemployment_change: Quarterly change in the unemployment rate in percentage points

In [7]:
df_economy["gdp_growth"] = df_economy["gdp"].pct_change(1)

In [8]:
df_economy["cpi_growth"] = df_economy["cpi"].pct_change(1)

In [9]:
df_economy["unemployment_change"] = df_economy["unemployment_rate"].diff()

In [10]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667
...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333


## Create YOY features

Since the dataset is quarterly, a lag of 4 quarters represents the same quarter in the previous year.

YoY features help identify annual changes while reducing the effect of seasonal patterns.

- gdp_yoy: GDP growth compared with the same quarter of the previous year
- cpi_yoy: CPI growth compared with the same quarter of the previous year
- unemployment_yoy_change: Year-over-year change in the unemployment rate

In [11]:
df_economy["gdp_yoy"] = df_economy["gdp"].pct_change(4)

In [12]:
df_economy["cpi_yoy"] = df_economy["cpi"].pct_change(4)

In [13]:
df_economy["unemployment_yoy_change"] = df_economy["unemployment_rate"].diff(4)

In [14]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,cpi_yoy,unemployment_yoy_change
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,NaN,NaN
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,NaN,NaN
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,NaN,NaN
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,0.111346,3.133333
...,...,...,...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667,0.071296,0.057498,-0.300000
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667,0.059455,0.040316,-0.066667
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333,0.062147,0.035618,0.166667
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333,0.058640,0.032362,0.166667


## Create rolling features

Rolling 4-quarter averages are used to smooth short-term fluctuations and capture the recent trend in economic variables.

- gdp_rolling_mean_4q: 4-quarter average of GDP growth
- cpi_rolling_mean_4q: 4-quarter average of CPI growth
- unemployment_rolling_mean_4q: 4-quarter average unemployment rate

In [15]:
df_economy["gdp_rolling_mean_4q"] = df_economy["gdp_growth"].rolling(4).mean()

In [16]:
df_economy["cpi_rolling_mean_4q"] = df_economy["cpi_growth"].rolling(4).mean()

In [17]:
df_economy["unemployment_rolling_mean_4q"] = df_economy["unemployment_rate"].rolling(4).mean()

In [18]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,cpi_yoy,unemployment_yoy_change,gdp_rolling_mean_4q,cpi_rolling_mean_4q,unemployment_rolling_mean_4q
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,NaN,NaN,NaN,NaN,NaN
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,NaN,NaN,NaN,NaN,NaN
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,NaN,NaN,NaN,NaN,5.641667
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,0.111346,3.133333,0.020333,0.026750,6.425000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667,0.071296,0.057498,-0.300000,0.017368,0.014092,3.558333
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667,0.059455,0.040316,-0.066667,0.014548,0.009932,3.541667
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333,0.062147,0.035618,0.166667,0.015195,0.008788,3.583333
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333,0.058640,0.032362,0.166667,0.014356,0.007994,3.625000


## Create volatility features

Rolling standard deviation is used as a simple measure of short-term economic volatility.

- gdp_volatility_4q: Volatility of GDP growth over the previous 4 quarters
- cpi_volatility_4q: Volatility of CPI growth over the previous 4 quarters
- unemployment_volatility_4q: Volatility of quarterly changes in unemployment over the previous 4 quarters

In [19]:
df_economy["gdp_volatility_4q"] = df_economy["gdp_growth"].rolling(4).std()

In [20]:
df_economy["cpi_volatility_4q"] = df_economy["cpi_growth"].rolling(4).std()

In [21]:
df_economy["unemployment_volatility_4q"] = df_economy["unemployment_change"].rolling(4).std()

In [22]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,cpi_yoy,unemployment_yoy_change,gdp_rolling_mean_4q,cpi_rolling_mean_4q,unemployment_rolling_mean_4q,gdp_volatility_4q,cpi_volatility_4q,unemployment_volatility_4q
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,NaN,NaN,NaN,NaN,5.641667,NaN,NaN,NaN
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,0.111346,3.133333,0.020333,0.026750,6.425000,0.007309,0.003937,0.695222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667,0.071296,0.057498,-0.300000,0.017368,0.014092,3.558333,0.002353,0.006909,0.083333
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667,0.059455,0.040316,-0.066667,0.014548,0.009932,3.541667,0.003630,0.002305,0.079349
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333,0.062147,0.035618,0.166667,0.015195,0.008788,3.583333,0.004490,0.001038,0.083333
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333,0.058640,0.032362,0.166667,0.014356,0.007994,3.625000,0.004622,0.001094,0.083333


## Create lag features

Lag features allow us to investigate whether changes in one economic variable are associated with changes in another variable in subsequent quarters.

- gdp_growth_lag1: GDP growth in the previous quarter
- gdp_growth_lag2: GDP growth two quarters earlier
- cpi_growth_lag1: CPI growth in the previous quarter
- unemployment_change_lag1: Unemployment change in the previous quarter

These features will be particularly useful when investigating delayed relationships between GDP growth, inflation, and unemployment.

In [23]:
df_economy["gdp_growth_lag1"] = df_economy["gdp_growth"].shift(1)

In [24]:
df_economy["gdp_growth_lag2"] = df_economy["gdp_growth"].shift(2)

In [25]:
df_economy["cpi_growth_lag1"] = df_economy["cpi_growth"].shift(1)

In [26]:
df_economy["unemployment_change_lag1"] = df_economy["unemployment_change"].shift(1)

In [27]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,...,gdp_rolling_mean_4q,cpi_rolling_mean_4q,unemployment_rolling_mean_4q,gdp_volatility_4q,cpi_volatility_4q,unemployment_volatility_4q,gdp_growth_lag1,gdp_growth_lag2,cpi_growth_lag1,unemployment_change_lag1
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.026051,NaN,0.026779,0.066667
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,...,NaN,NaN,5.641667,NaN,NaN,NaN,0.019588,0.026051,0.028140,0.433333
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,...,0.020333,0.026750,6.425000,0.007309,0.003937,0.695222,0.025418,0.019588,0.030708,0.966667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667,0.071296,...,0.017368,0.014092,3.558333,0.002353,0.006909,0.083333,0.015917,0.017631,0.009922,0.033333
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667,0.059455,...,0.014548,0.009932,3.541667,0.003630,0.002305,0.079349,0.015343,0.015917,0.009255,-0.066667
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333,0.062147,...,0.015195,0.008788,3.583333,0.004490,0.001038,0.083333,0.009302,0.015343,0.007515,0.066667
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333,0.058640,...,0.014356,0.007994,3.625000,0.004622,0.001094,0.083333,0.020216,0.009302,0.008461,0.133333


## Create economic flags

Binary indicators are created to identify economically important periods and observations.


### Covid flag
identifies the period from 2020Q1 through 2021Q1 for analysis of the economic impact of the COVID-19 pandemic.

In [28]:
df_economy["covid_flag"] = df_economy["year-quarter"].between(left="2020Q1",right="2021Q1",inclusive="both")

### Resession flag
identifies quarters where:

- GDP growth < 0
- Unemployment change > 0

This is a simplified analytical indicator created for this project and is not an official recession classification.

In [29]:
df_economy["recession_flag"] = (
    (df_economy["gdp_growth"] < 0) &
    (df_economy["unemployment_change"] > 0)
)

### High growth flag
identifies quarters where GDP growth is above the 75th percentile of the observed GDP growth distribution.

In [30]:
threshold = df_economy["gdp_growth"].quantile(0.75)

df_economy["high_growth_flag"] = (df_economy["gdp_growth"] > threshold)

In [31]:
df_economy[
    ["year-quarter", "gdp_growth", "unemployment_change",
     "covid_flag", "recession_flag", "high_growth_flag"]
].head(20)

,year-quarter,gdp_growth,unemployment_change,covid_flag,recession_flag,high_growth_flag
0,1974Q1,NaN,NaN,False,False,False
1,1974Q2,0.026051,0.066667,False,False,True
2,1974Q3,0.019588,0.433333,False,False,True
3,1974Q4,0.025418,0.966667,False,False,True
4,1975Q1,0.010275,1.666667,False,False,False
5,1975Q2,0.022113,0.600000,False,False,True
6,1975Q3,0.035092,-0.400000,False,False,True
7,1975Q4,0.030419,-0.166667,False,False,True
8,1976Q1,0.033293,-0.566667,False,False,True
9,1976Q2,0.017493,-0.166667,False,False,False


In [32]:
df_economy[
    ["covid_flag", "recession_flag", "high_growth_flag"]
].sum()

covid_flag           5
recession_flag       9
high_growth_flag    50
dtype: int64

## Checking resulting dataframe

The resulting dataset is checked for:

- Number of observations and features
- Data types
- Missing values
- Correct time ordering
- Duplicate observations

In [33]:
df_economy.shape

(201, 25)

In [34]:
df_economy.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   year                          201 non-null    int32  
 1   quarter                       201 non-null    int32  
 2   year-quarter                  201 non-null    str    
 3   gdp                           201 non-null    float64
 4   cpi                           201 non-null    float64
 5   unemployment_rate             201 non-null    float64
 6   gdp_growth                    200 non-null    float64
 7   cpi_growth                    200 non-null    float64
 8   unemployment_change           200 non-null    float64
 9   gdp_yoy                       197 non-null    float64
 10  cpi_yoy                       197 non-null    float64
 11  unemployment_yoy_change       197 non-null    float64
 12  gdp_rolling_mean_4q           197 non-null    float64
 13  cpi_rolling_mean

In [35]:
df_economy[
    [
        "gdp_growth",
        "cpi_growth",
        "unemployment_change",
        "gdp_yoy",
        "cpi_yoy",
        "unemployment_yoy_change",
        "gdp_rolling_mean_4q",
        "cpi_rolling_mean_4q",
        "unemployment_rolling_mean_4q",
        "gdp_volatility_4q",
        "cpi_volatility_4q",
        "unemployment_volatility_4q",
        "gdp_growth_lag1",
        "gdp_growth_lag2",
        "cpi_growth_lag1",
        "unemployment_change_lag1",
        "covid_flag",
        "recession_flag",
        "high_growth_flag"
    ]
].head()

,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,cpi_yoy,unemployment_yoy_change,gdp_rolling_mean_4q,cpi_rolling_mean_4q,unemployment_rolling_mean_4q,gdp_volatility_4q,cpi_volatility_4q,unemployment_volatility_4q,gdp_growth_lag1,gdp_growth_lag2,cpi_growth_lag1,unemployment_change_lag1,covid_flag,recession_flag,high_growth_flag
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
1,0.026051,0.026779,0.066667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,True
2,0.019588,0.028140,0.433333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.026051,NaN,0.026779,0.066667,False,False,True
3,0.025418,0.030708,0.966667,NaN,NaN,NaN,NaN,NaN,5.641667,NaN,NaN,NaN,0.019588,0.026051,0.028140,0.433333,False,False,True
4,0.010275,0.021373,1.666667,0.083762,0.111346,3.133333,0.020333,0.02675,6.425000,0.007309,0.003937,0.695222,0.025418,0.019588,0.030708,0.966667,False,False,False


In [36]:
df_economy.isnull().sum()

year                            0
quarter                         0
year-quarter                    0
gdp                             0
cpi                             0
unemployment_rate               0
gdp_growth                      1
cpi_growth                      1
unemployment_change             1
gdp_yoy                         4
cpi_yoy                         4
unemployment_yoy_change         4
gdp_rolling_mean_4q             4
cpi_rolling_mean_4q             4
unemployment_rolling_mean_4q    3
gdp_volatility_4q               4
cpi_volatility_4q               4
unemployment_volatility_4q      4
gdp_growth_lag1                 2
gdp_growth_lag2                 3
cpi_growth_lag1                 2
unemployment_change_lag1        2
covid_flag                      0
recession_flag                  0
high_growth_flag                0
dtype: int64

In [37]:
df_economy.head()

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,...,gdp_volatility_4q,cpi_volatility_4q,unemployment_volatility_4q,gdp_growth_lag1,gdp_growth_lag2,cpi_growth_lag1,unemployment_change_lag1,covid_flag,recession_flag,high_growth_flag
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,True
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,...,NaN,NaN,NaN,0.026051,NaN,0.026779,0.066667,False,False,True
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,...,NaN,NaN,NaN,0.019588,0.026051,0.028140,0.433333,False,False,True
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,...,0.007309,0.003937,0.695222,0.025418,0.019588,0.030708,0.966667,False,False,False


## Checking missing values

Missing values in engineered features are expected at the beginning of the dataset because some calculations require previous observations.

For example:
- Growth features require the previous quarter.
- YoY features require four previous quarters.
- Rolling features require a sufficient number of observations.
- Lag features require previous observations.

These are structural missing values rather than missing observations in the original dataset.

In [38]:
missing = df_economy.isnull().sum()
missing[missing > 0]

gdp_growth                      1
cpi_growth                      1
unemployment_change             1
gdp_yoy                         4
cpi_yoy                         4
unemployment_yoy_change         4
gdp_rolling_mean_4q             4
cpi_rolling_mean_4q             4
unemployment_rolling_mean_4q    3
gdp_volatility_4q               4
cpi_volatility_4q               4
unemployment_volatility_4q      4
gdp_growth_lag1                 2
gdp_growth_lag2                 3
cpi_growth_lag1                 2
unemployment_change_lag1        2
dtype: int64

### Check data types
Check that each variable has an appropriate data type.

In [39]:
df_economy.dtypes

year                              int32
quarter                           int32
year-quarter                        str
gdp                             float64
cpi                             float64
unemployment_rate               float64
gdp_growth                      float64
cpi_growth                      float64
unemployment_change             float64
gdp_yoy                         float64
cpi_yoy                         float64
unemployment_yoy_change         float64
gdp_rolling_mean_4q             float64
cpi_rolling_mean_4q             float64
unemployment_rolling_mean_4q    float64
gdp_volatility_4q               float64
cpi_volatility_4q               float64
unemployment_volatility_4q      float64
gdp_growth_lag1                 float64
gdp_growth_lag2                 float64
cpi_growth_lag1                 float64
unemployment_change_lag1        float64
covid_flag                         bool
recession_flag                     bool
high_growth_flag                   bool


In [40]:
df_economy.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   year                          201 non-null    int32  
 1   quarter                       201 non-null    int32  
 2   year-quarter                  201 non-null    str    
 3   gdp                           201 non-null    float64
 4   cpi                           201 non-null    float64
 5   unemployment_rate             201 non-null    float64
 6   gdp_growth                    200 non-null    float64
 7   cpi_growth                    200 non-null    float64
 8   unemployment_change           200 non-null    float64
 9   gdp_yoy                       197 non-null    float64
 10  cpi_yoy                       197 non-null    float64
 11  unemployment_yoy_change       197 non-null    float64
 12  gdp_rolling_mean_4q           197 non-null    float64
 13  cpi_rolling_mean

### Check time sorting

Verify that quarterly observations are ordered chronologically.

In [41]:
df_economy["year-quarter"].is_monotonic_increasing

True

In [42]:
df_economy[["year", "quarter", "year-quarter"]].head()

,year,quarter,year-quarter
0,1974,1,1974Q1
1,1974,2,1974Q2
2,1974,3,1974Q3
3,1974,4,1974Q4
4,1975,1,1975Q1


In [43]:
df_economy[["year", "quarter", "year-quarter"]].tail()

,year,quarter,year-quarter
196,2023,1,2023Q1
197,2023,2,2023Q2
198,2023,3,2023Q3
199,2023,4,2023Q4
200,2024,1,2024Q1


### Check duplicate rows

Check for duplicate rows and duplicate quarter identifiers.

Each quarter should have only one observation.

In [44]:
df_economy.duplicated().sum()

np.int64(0)

In [45]:
df_economy["year-quarter"].duplicated().sum()

np.int64(0)

## Save processed dataset
After completing feature engineering and data quality checks, the final feature-engineered dataset is saved as a CSV file.

This dataset will be used as the input for the exploratory analysis in the next notebook.

In [46]:
PROCESSED_PATH = Path("../data/processed")
df_economy.to_csv(PROCESSED_PATH / "USA_economy.csv", index=False)

## Objective

The goal of this notebook is to create additional variables from the cleaned quarterly US economic dataset.

These engineered features will be used later for exploratory and economic analysis.

Main feature groups:
- Growth features
- Year-over-Year (YoY) features
- Rolling statistics
- Volatility measures
- Lag features
- Economic period flags

## Load df_economy

In [47]:
PROCESSED_PATH = Path("../data/processed")

df_economy= pd.read_feather(
    PROCESSED_PATH / "economy_quarterly.feather"
)

## Data observation

In [48]:
df_economy.head()

,year,quarter,year-quarter,gdp,cpi,unemployment_rate
0,1974,1,1974Q1,1491.209,47.300000,5.133333
1,1974,2,1974Q2,1530.056,48.566667,5.200000
2,1974,3,1974Q3,1560.026,49.933333,5.633333
3,1974,4,1974Q4,1599.679,51.466667,6.600000
4,1975,1,1975Q1,1616.116,52.566667,8.266667


In [49]:
df_economy.tail()

,year,quarter,year-quarter,gdp,cpi,unemployment_rate
196,2023,1,2023Q1,26813.601,301.203000,3.500000
197,2023,2,2023Q2,27063.012,303.466667,3.566667
198,2023,3,2023Q3,27610.128,306.034333,3.700000
199,2023,4,2023Q4,27956.998,308.099000,3.733333
200,2024,1,2024Q1,28269.174,309.685000,3.700000


In [50]:
df_economy.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               201 non-null    int32  
 1   quarter            201 non-null    int32  
 2   year-quarter       201 non-null    str    
 3   gdp                201 non-null    float64
 4   cpi                201 non-null    float64
 5   unemployment_rate  201 non-null    float64
dtypes: float64(3), int32(2), str(1)
memory usage: 9.2 KB


## Create growth features

Growth features measure quarter-to-quarter changes in the main economic variables.

- gdp_growth: Quarterly GDP growth rate
- cpi_growth: Quarterly CPI growth rate
- unemployment_change: Quarterly change in the unemployment rate in percentage points

In [51]:
df_economy["gdp_growth"] = df_economy["gdp"].pct_change(1)

In [52]:
df_economy["cpi_growth"] = df_economy["cpi"].pct_change(1)

In [53]:
df_economy["unemployment_change"] = df_economy["unemployment_rate"].diff()

In [54]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667
...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333


## Create YOY features

Since the dataset is quarterly, a lag of 4 quarters represents the same quarter in the previous year.

YoY features help identify annual changes while reducing the effect of seasonal patterns.

- gdp_yoy: GDP growth compared with the same quarter of the previous year
- cpi_yoy: CPI growth compared with the same quarter of the previous year
- unemployment_yoy_change: Year-over-year change in the unemployment rate

In [55]:
df_economy["gdp_yoy"] = df_economy["gdp"].pct_change(4)

In [56]:
df_economy["cpi_yoy"] = df_economy["cpi"].pct_change(4)

In [57]:
df_economy["unemployment_yoy_change"] = df_economy["unemployment_rate"].diff(4)

In [58]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,cpi_yoy,unemployment_yoy_change
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,NaN,NaN
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,NaN,NaN
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,NaN,NaN
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,0.111346,3.133333
...,...,...,...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667,0.071296,0.057498,-0.300000
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667,0.059455,0.040316,-0.066667
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333,0.062147,0.035618,0.166667
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333,0.058640,0.032362,0.166667


## Create rolling features

Rolling 4-quarter averages are used to smooth short-term fluctuations and capture the recent trend in economic variables.

- gdp_rolling_mean_4q: 4-quarter average of GDP growth
- cpi_rolling_mean_4q: 4-quarter average of CPI growth
- unemployment_rolling_mean_4q: 4-quarter average unemployment rate

In [59]:
df_economy["gdp_rolling_mean_4q"] = df_economy["gdp_growth"].rolling(4).mean()

In [60]:
df_economy["cpi_rolling_mean_4q"] = df_economy["cpi_growth"].rolling(4).mean()

In [61]:
df_economy["unemployment_rolling_mean_4q"] = df_economy["unemployment_rate"].rolling(4).mean()

In [62]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,cpi_yoy,unemployment_yoy_change,gdp_rolling_mean_4q,cpi_rolling_mean_4q,unemployment_rolling_mean_4q
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,NaN,NaN,NaN,NaN,NaN
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,NaN,NaN,NaN,NaN,NaN
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,NaN,NaN,NaN,NaN,5.641667
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,0.111346,3.133333,0.020333,0.026750,6.425000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667,0.071296,0.057498,-0.300000,0.017368,0.014092,3.558333
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667,0.059455,0.040316,-0.066667,0.014548,0.009932,3.541667
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333,0.062147,0.035618,0.166667,0.015195,0.008788,3.583333
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333,0.058640,0.032362,0.166667,0.014356,0.007994,3.625000


## Create volatility features

Rolling standard deviation is used as a simple measure of short-term economic volatility.

- gdp_volatility_4q: Volatility of GDP growth over the previous 4 quarters
- cpi_volatility_4q: Volatility of CPI growth over the previous 4 quarters
- unemployment_volatility_4q: Volatility of quarterly changes in unemployment over the previous 4 quarters

In [63]:
df_economy["gdp_volatility_4q"] = df_economy["gdp_growth"].rolling(4).std()

In [64]:
df_economy["cpi_volatility_4q"] = df_economy["cpi_growth"].rolling(4).std()

In [65]:
df_economy["unemployment_volatility_4q"] = df_economy["unemployment_change"].rolling(4).std()

In [66]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,cpi_yoy,unemployment_yoy_change,gdp_rolling_mean_4q,cpi_rolling_mean_4q,unemployment_rolling_mean_4q,gdp_volatility_4q,cpi_volatility_4q,unemployment_volatility_4q
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,NaN,NaN,NaN,NaN,5.641667,NaN,NaN,NaN
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,0.111346,3.133333,0.020333,0.026750,6.425000,0.007309,0.003937,0.695222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667,0.071296,0.057498,-0.300000,0.017368,0.014092,3.558333,0.002353,0.006909,0.083333
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667,0.059455,0.040316,-0.066667,0.014548,0.009932,3.541667,0.003630,0.002305,0.079349
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333,0.062147,0.035618,0.166667,0.015195,0.008788,3.583333,0.004490,0.001038,0.083333
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333,0.058640,0.032362,0.166667,0.014356,0.007994,3.625000,0.004622,0.001094,0.083333


## Create lag features

Lag features allow us to investigate whether changes in one economic variable are associated with changes in another variable in subsequent quarters.

- gdp_growth_lag1: GDP growth in the previous quarter
- gdp_growth_lag2: GDP growth two quarters earlier
- cpi_growth_lag1: CPI growth in the previous quarter
- unemployment_change_lag1: Unemployment change in the previous quarter

These features will be particularly useful when investigating delayed relationships between GDP growth, inflation, and unemployment.

In [67]:
df_economy["gdp_growth_lag1"] = df_economy["gdp_growth"].shift(1)

In [68]:
df_economy["gdp_growth_lag2"] = df_economy["gdp_growth"].shift(2)

In [69]:
df_economy["cpi_growth_lag1"] = df_economy["cpi_growth"].shift(1)

In [70]:
df_economy["unemployment_change_lag1"] = df_economy["unemployment_change"].shift(1)

In [71]:
df_economy

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,...,gdp_rolling_mean_4q,cpi_rolling_mean_4q,unemployment_rolling_mean_4q,gdp_volatility_4q,cpi_volatility_4q,unemployment_volatility_4q,gdp_growth_lag1,gdp_growth_lag2,cpi_growth_lag1,unemployment_change_lag1
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.026051,NaN,0.026779,0.066667
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,...,NaN,NaN,5.641667,NaN,NaN,NaN,0.019588,0.026051,0.028140,0.433333
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,...,0.020333,0.026750,6.425000,0.007309,0.003937,0.695222,0.025418,0.019588,0.030708,0.966667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,2023,1,2023Q1,26813.601,301.203000,3.500000,0.015343,0.009255,-0.066667,0.071296,...,0.017368,0.014092,3.558333,0.002353,0.006909,0.083333,0.015917,0.017631,0.009922,0.033333
197,2023,2,2023Q2,27063.012,303.466667,3.566667,0.009302,0.007515,0.066667,0.059455,...,0.014548,0.009932,3.541667,0.003630,0.002305,0.079349,0.015343,0.015917,0.009255,-0.066667
198,2023,3,2023Q3,27610.128,306.034333,3.700000,0.020216,0.008461,0.133333,0.062147,...,0.015195,0.008788,3.583333,0.004490,0.001038,0.083333,0.009302,0.015343,0.007515,0.066667
199,2023,4,2023Q4,27956.998,308.099000,3.733333,0.012563,0.006747,0.033333,0.058640,...,0.014356,0.007994,3.625000,0.004622,0.001094,0.083333,0.020216,0.009302,0.008461,0.133333


## Create economic flags

Binary indicators are created to identify economically important periods and observations.


### Covid flag
identifies the period from 2020Q1 through 2021Q1 for analysis of the economic impact of the COVID-19 pandemic.

In [72]:
df_economy["covid_flag"] = df_economy["year-quarter"].between(left="2020Q1",right="2021Q1",inclusive="both")

### Resession flag
identifies quarters where:

- GDP growth < 0
- Unemployment change > 0

This is a simplified analytical indicator created for this project and is not an official recession classification.

In [73]:
df_economy["recession_flag"] = (
    (df_economy["gdp_growth"] < 0) &
    (df_economy["unemployment_change"] > 0)
)

### High growth flag
identifies quarters where GDP growth is above the 75th percentile of the observed GDP growth distribution.

In [74]:
threshold = df_economy["gdp_growth"].quantile(0.75)

df_economy["high_growth_flag"] = (df_economy["gdp_growth"] > threshold)

In [75]:
df_economy[
    ["year-quarter", "gdp_growth", "unemployment_change",
     "covid_flag", "recession_flag", "high_growth_flag"]
].head(20)

,year-quarter,gdp_growth,unemployment_change,covid_flag,recession_flag,high_growth_flag
0,1974Q1,NaN,NaN,False,False,False
1,1974Q2,0.026051,0.066667,False,False,True
2,1974Q3,0.019588,0.433333,False,False,True
3,1974Q4,0.025418,0.966667,False,False,True
4,1975Q1,0.010275,1.666667,False,False,False
5,1975Q2,0.022113,0.600000,False,False,True
6,1975Q3,0.035092,-0.400000,False,False,True
7,1975Q4,0.030419,-0.166667,False,False,True
8,1976Q1,0.033293,-0.566667,False,False,True
9,1976Q2,0.017493,-0.166667,False,False,False


In [76]:
df_economy[
    ["covid_flag", "recession_flag", "high_growth_flag"]
].sum()

covid_flag           5
recession_flag       9
high_growth_flag    50
dtype: int64

## Checking resulting dataframe

The resulting dataset is checked for:

- Number of observations and features
- Data types
- Missing values
- Correct time ordering
- Duplicate observations

In [77]:
df_economy.shape

(201, 25)

In [78]:
df_economy.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   year                          201 non-null    int32  
 1   quarter                       201 non-null    int32  
 2   year-quarter                  201 non-null    str    
 3   gdp                           201 non-null    float64
 4   cpi                           201 non-null    float64
 5   unemployment_rate             201 non-null    float64
 6   gdp_growth                    200 non-null    float64
 7   cpi_growth                    200 non-null    float64
 8   unemployment_change           200 non-null    float64
 9   gdp_yoy                       197 non-null    float64
 10  cpi_yoy                       197 non-null    float64
 11  unemployment_yoy_change       197 non-null    float64
 12  gdp_rolling_mean_4q           197 non-null    float64
 13  cpi_rolling_mean

In [79]:
df_economy[
    [
        "gdp_growth",
        "cpi_growth",
        "unemployment_change",
        "gdp_yoy",
        "cpi_yoy",
        "unemployment_yoy_change",
        "gdp_rolling_mean_4q",
        "cpi_rolling_mean_4q",
        "unemployment_rolling_mean_4q",
        "gdp_volatility_4q",
        "cpi_volatility_4q",
        "unemployment_volatility_4q",
        "gdp_growth_lag1",
        "gdp_growth_lag2",
        "cpi_growth_lag1",
        "unemployment_change_lag1",
        "covid_flag",
        "recession_flag",
        "high_growth_flag"
    ]
].head()

,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,cpi_yoy,unemployment_yoy_change,gdp_rolling_mean_4q,cpi_rolling_mean_4q,unemployment_rolling_mean_4q,gdp_volatility_4q,cpi_volatility_4q,unemployment_volatility_4q,gdp_growth_lag1,gdp_growth_lag2,cpi_growth_lag1,unemployment_change_lag1,covid_flag,recession_flag,high_growth_flag
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
1,0.026051,0.026779,0.066667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,True
2,0.019588,0.028140,0.433333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.026051,NaN,0.026779,0.066667,False,False,True
3,0.025418,0.030708,0.966667,NaN,NaN,NaN,NaN,NaN,5.641667,NaN,NaN,NaN,0.019588,0.026051,0.028140,0.433333,False,False,True
4,0.010275,0.021373,1.666667,0.083762,0.111346,3.133333,0.020333,0.02675,6.425000,0.007309,0.003937,0.695222,0.025418,0.019588,0.030708,0.966667,False,False,False


In [80]:
df_economy.isnull().sum()

year                            0
quarter                         0
year-quarter                    0
gdp                             0
cpi                             0
unemployment_rate               0
gdp_growth                      1
cpi_growth                      1
unemployment_change             1
gdp_yoy                         4
cpi_yoy                         4
unemployment_yoy_change         4
gdp_rolling_mean_4q             4
cpi_rolling_mean_4q             4
unemployment_rolling_mean_4q    3
gdp_volatility_4q               4
cpi_volatility_4q               4
unemployment_volatility_4q      4
gdp_growth_lag1                 2
gdp_growth_lag2                 3
cpi_growth_lag1                 2
unemployment_change_lag1        2
covid_flag                      0
recession_flag                  0
high_growth_flag                0
dtype: int64

In [81]:
df_economy.head()

,year,quarter,year-quarter,gdp,cpi,unemployment_rate,gdp_growth,cpi_growth,unemployment_change,gdp_yoy,...,gdp_volatility_4q,cpi_volatility_4q,unemployment_volatility_4q,gdp_growth_lag1,gdp_growth_lag2,cpi_growth_lag1,unemployment_change_lag1,covid_flag,recession_flag,high_growth_flag
0,1974,1,1974Q1,1491.209,47.300000,5.133333,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
1,1974,2,1974Q2,1530.056,48.566667,5.200000,0.026051,0.026779,0.066667,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,True
2,1974,3,1974Q3,1560.026,49.933333,5.633333,0.019588,0.028140,0.433333,NaN,...,NaN,NaN,NaN,0.026051,NaN,0.026779,0.066667,False,False,True
3,1974,4,1974Q4,1599.679,51.466667,6.600000,0.025418,0.030708,0.966667,NaN,...,NaN,NaN,NaN,0.019588,0.026051,0.028140,0.433333,False,False,True
4,1975,1,1975Q1,1616.116,52.566667,8.266667,0.010275,0.021373,1.666667,0.083762,...,0.007309,0.003937,0.695222,0.025418,0.019588,0.030708,0.966667,False,False,False


## Checking missing values

Missing values in engineered features are expected at the beginning of the dataset because some calculations require previous observations.

For example:
- Growth features require the previous quarter.
- YoY features require four previous quarters.
- Rolling features require a sufficient number of observations.
- Lag features require previous observations.

These are structural missing values rather than missing observations in the original dataset.

In [82]:
missing = df_economy.isnull().sum()
missing[missing > 0]

gdp_growth                      1
cpi_growth                      1
unemployment_change             1
gdp_yoy                         4
cpi_yoy                         4
unemployment_yoy_change         4
gdp_rolling_mean_4q             4
cpi_rolling_mean_4q             4
unemployment_rolling_mean_4q    3
gdp_volatility_4q               4
cpi_volatility_4q               4
unemployment_volatility_4q      4
gdp_growth_lag1                 2
gdp_growth_lag2                 3
cpi_growth_lag1                 2
unemployment_change_lag1        2
dtype: int64

## Data quality check

### Check data types
Check that each variable has an appropriate data type.

In [83]:
df_economy.dtypes

year                              int32
quarter                           int32
year-quarter                        str
gdp                             float64
cpi                             float64
unemployment_rate               float64
gdp_growth                      float64
cpi_growth                      float64
unemployment_change             float64
gdp_yoy                         float64
cpi_yoy                         float64
unemployment_yoy_change         float64
gdp_rolling_mean_4q             float64
cpi_rolling_mean_4q             float64
unemployment_rolling_mean_4q    float64
gdp_volatility_4q               float64
cpi_volatility_4q               float64
unemployment_volatility_4q      float64
gdp_growth_lag1                 float64
gdp_growth_lag2                 float64
cpi_growth_lag1                 float64
unemployment_change_lag1        float64
covid_flag                         bool
recession_flag                     bool
high_growth_flag                   bool


In [84]:
df_economy.info()

<class 'pandas.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 25 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   year                          201 non-null    int32  
 1   quarter                       201 non-null    int32  
 2   year-quarter                  201 non-null    str    
 3   gdp                           201 non-null    float64
 4   cpi                           201 non-null    float64
 5   unemployment_rate             201 non-null    float64
 6   gdp_growth                    200 non-null    float64
 7   cpi_growth                    200 non-null    float64
 8   unemployment_change           200 non-null    float64
 9   gdp_yoy                       197 non-null    float64
 10  cpi_yoy                       197 non-null    float64
 11  unemployment_yoy_change       197 non-null    float64
 12  gdp_rolling_mean_4q           197 non-null    float64
 13  cpi_rolling_mean

### Check time sorting

Verify that quarterly observations are ordered chronologically.

In [85]:
df_economy["year-quarter"].is_monotonic_increasing

True

In [86]:
df_economy[["year", "quarter", "year-quarter"]].head()

,year,quarter,year-quarter
0,1974,1,1974Q1
1,1974,2,1974Q2
2,1974,3,1974Q3
3,1974,4,1974Q4
4,1975,1,1975Q1


In [87]:
df_economy[["year", "quarter", "year-quarter"]].tail()

,year,quarter,year-quarter
196,2023,1,2023Q1
197,2023,2,2023Q2
198,2023,3,2023Q3
199,2023,4,2023Q4
200,2024,1,2024Q1


### Check duplicate rows

Check for duplicate rows and duplicate quarter identifiers.

Each quarter should have only one observation.

In [88]:
df_economy.duplicated().sum()

np.int64(0)

In [89]:
df_economy["year-quarter"].duplicated().sum()

np.int64(0)

## Save processed dataset
After completing feature engineering and data quality checks, the final feature-engineered dataset is saved as a CSV file.

This dataset will be used as the input for the exploratory analysis in the next notebook.

In [90]:
PROCESSED_PATH = Path("../data/processed")
df_economy.to_csv(PROCESSED_PATH / "USA_economy.csv", index=False)